# 14. Revisión de literatura y declaración de novedad

La guía del Entregable 3 exige que la novedad del modelo original **se demuestre** con una revisión documentada (sección 5.2): al menos tres bases de datos, al menos cinco cadenas de búsqueda con su fecha, una matriz de novedad, búsqueda por citación hacia atrás y hacia adelante, revisión de repositorios de código y una declaración de novedad que nombre los trabajos más cercanos y en qué se diferencia la propuesta.

Este capítulo documenta esa revisión para **DES-CPL** (capítulo 13), en cinco bases:
- **Cuatro con API pública.** `experiments/revision_literatura.py` ejecuta las mismas cadenas contra ellas y guarda los resultados con la fecha de la consulta en `results/literatura/`.
- **Google Scholar.** No tiene API, así que las páginas de resultados se copiaron del navegador a `results/literatura/google_scholar/` y se leen con `src/literatura.py`.

In [1]:
from config import *
from IPython.display import display

LITERATURA = RESULTADOS / "literatura"
fechas = sorted(p.stem.split("_")[1] for p in LITERATURA.glob("busquedas_*.csv"))
FECHA = fechas[-1]
busquedas = pd.read_csv(LITERATURA / f"busquedas_{FECHA}.csv")
registros = pd.read_csv(LITERATURA / f"registros_{FECHA}.csv")
print(f"Consultas por API disponibles: {fechas}. Se usa la del {FECHA}.")

# Google Scholar: páginas de resultados copiadas del navegador, una por cadena.
from src.literatura import leer_google_scholar
CARPETA_GS = LITERATURA / "google_scholar"
if CARPETA_GS.exists():
    consultas_gs = {"C1": '"dynamic ensemble selection" readmission',
                    "C2": '"dynamic classifier selection" readmission diabetes',
                    "C3": '"dynamic ensemble selection" OR "dynamic classifier selection" diabetes',
                    "C4": '"130-US hospitals" "dynamic selection"',
                    "C5": '"selección dinámica" clasificadores readmisión diabetes',
                    "C6": '"region of competence" readmission diabetes',
                    "C7": '"dynamic ensemble selection" shrinkage "Brier score"',
                    "C8": '"dynamic ensemble selection" OR "dynamic classifier selection" readmission'}
    busquedas_gs, registros_gs = leer_google_scholar(CARPETA_GS, consultas_gs)
    busquedas_gs["descripción"] = busquedas_gs["cadena"].map(
        busquedas.drop_duplicates("cadena").set_index("cadena")["descripción"])
    print("Google Scholar:", dict(zip(busquedas_gs["cadena"], busquedas_gs["fecha"])))
    busquedas = pd.concat([busquedas, busquedas_gs], ignore_index=True)
    registros = pd.concat([registros, registros_gs], ignore_index=True)

Consultas por API disponibles: ['2026-11-10']. Se usa la del 2026-11-10.
Google Scholar: {'C1': '2026-11-10', 'C2': '2026-11-10', 'C3': '2026-11-10', 'C4': '2026-11-10', 'C5': '2026-11-10', 'C6': '2026-11-10', 'C7': '2026-11-10', 'C8': '2026-11-10'}


## 14.1. Pregunta y criterios

**Pregunta.** ¿Existe algún estudio que aplique selección dinámica de clasificadores o de ensambles (DCS o DES) a la predicción de la readmisión hospitalaria de pacientes con diabetes, o que proponga la misma regla de combinación de DES-CPL?

Se considera **relevante para la novedad** un trabajo que cumpla al menos uno de estos criterios:

| Criterio | Por qué podría invalidar la novedad |
|---|---|
| A. Selección dinámica aplicada a readmisión hospitalaria (cualquier enfermedad) | Mismo tipo de modelo en el mismo tipo de problema |
| B. Selección dinámica aplicada a pacientes con diabetes (cualquier desenlace) | Mismo tipo de modelo en la misma población |
| C. Un método de selección dinámica con alguno de los componentes de DES-CPL (competencia por Brier con prevalencia local, región estratificada en el espacio de logits, encogimiento local-global) | Mismo método en otro problema |
| D. Un enfoque local por similitud entre pacientes evaluado en el conjunto Diabetes 130-US | Misma idea (vecindarios de pacientes) en los mismos datos |

**Exclusiones.** Redes neuronales como modelo propuesto (la guía las excluye del modelo original), aunque se registran si cumplen el criterio A. El conjunto «diabetes» de los bancos de prueba de selección dinámica (Pima) es de *diagnóstico* de diabetes, no de readmisión, y cuenta como criterio B solo si el trabajo lo plantea como aplicación clínica.

**Proceso de revisión.** Se leyeron el título de todos los registros devueltos y el resumen de los que no se descartaron por título. Se consultó el texto completo, cuando era de acceso abierto, de los trabajos marcados como relevantes.

## 14.2. Bases y cadenas de búsqueda

Se consultaron cinco bases. Cuatro tienen API pública, lo que permite repetir la búsqueda exactamente; Google Scholar se consultó a mano en el navegador:

| Base | Cobertura | Campo buscado |
|---|---|---|
| **OpenAlex** | Índice multidisciplinario abierto (sucesor de Microsoft Academic; incluye Crossref, PubMed y repositorios) | Título y resumen |
| **PubMed** | Literatura biomédica (MEDLINE y PubMed Central) | `[tiab]`, título y resumen, con truncamiento `*` |
| **arXiv** | Preprints de aprendizaje automático y estadística | Resumen (`abs:`) |
| **Semantic Scholar** | Índice de literatura científica con énfasis en computación y biomedicina | Título y resumen |
| **Google Scholar** | El índice académico más amplio; incluye tesis, libros y literatura gris | **Texto completo** |

Las ocho cadenas combinan (i) el nombre o la familia del modelo, (ii) el problema y (iii) sinónimos; C5 es la versión en español. La sintaxis cambia por base (por ejemplo, Semantic Scholar usa `+` para AND y `|` para OR), pero los términos son los mismos. La tabla muestra la versión de OpenAlex; las demás están en el script y en `busquedas_<fecha>.csv`.

**Cómo se consultó Google Scholar.** Google Scholar bloquea las consultas automatizadas (devolvió el error 429 al intentarlo). Por eso se consultó a mano:
- Configuración: cualquier idioma, 20 resultados por página, sin patentes ni citas.
- Las cadenas se adaptaron a su sintaxis; las comillas y `OR` funcionan, pero no los paréntesis.
- Cada página de resultados se copió completa a `google_scholar/<cadena>.txt`.
- `src/literatura.py` extrae de esos archivos el total de resultados y, por registro, título, autores, año, fuente y fragmento.

Como Google Scholar busca en el texto completo, devuelve muchos trabajos que solo mencionan los términos en las referencias.

Web of Science y Scopus requieren licencia institucional y se usaron en la búsqueda por problema de la sección 14.6.

In [2]:
cadenas = (busquedas.query("base == 'OpenAlex'")
           .loc[:, ["cadena", "descripción", "consulta"]]
           .rename(columns={"consulta": "consulta (OpenAlex)"})
           .set_index("cadena"))
with pd.option_context("display.max_colwidth", None):
    display(cadenas)

,descripción,consulta (OpenAlex)
cadena,,
C1,Selección dinámica de ensambles + readmisión,"""dynamic ensemble selection"" AND (readmission OR rehospitalization OR readmitted)"
C2,Selección dinámica de clasificadores + readmisión + diabetes,"(""dynamic classifier selection"" OR ""dynamic selection"") AND (readmission OR rehospitalization) AND (diabetes OR diabetic)"
C3,"Métodos de selección dinámica (KNORA, META-DES) + diabetes","(""dynamic ensemble selection"" OR ""dynamic classifier selection"" OR KNORA OR ""META-DES"") AND (diabetes OR diabetic)"
C4,Conjunto Diabetes 130-US + ensambles o selección dinámica,"(""130-US hospitals"" OR ""130 US hospitals"" OR ""Diabetes 130"") AND (ensemble OR ""dynamic selection"")"
C5,Cadena en español: selección dinámica + readmisión o diabetes,"(""selección dinámica"" OR ""seleccion dinamica"") AND (clasificadores OR ensambles OR ensamble) AND (readmisión OR reingreso OR diabetes)"
C6,Región o competencia local + readmisión o diabetes,"(""local competence"" OR ""region of competence"" OR ""competence region"") AND (readmission OR diabetes OR diabetic)"
C7,"Componentes de DES-CPL: encogimiento, competencia probabilística, perfil de salidas","""dynamic ensemble selection"" AND (shrinkage OR ""probabilistic competence"" OR ""output profile"" OR ""output profiles"")"
C8,Cualquier selección dinámica + readmisión (sin restringir enfermedad),"(""dynamic ensemble selection"" OR ""dynamic classifier selection"" OR ""dynamic selection"") AND (readmission OR rehospitalization OR readmitted)"


## 14.3. Matriz de novedad

La matriz sigue la plantilla de la guía (Tabla 2): para cada base y cadena, el número de resultados, cuántos se revisaron, cuántos resultaron relevantes según los criterios de 14.1 y una observación. Los registros relevantes se identificaron leyendo título y resumen; su lista está en la celda siguiente, con el criterio que cumplen.

In [3]:
def normalizar(titulo):
    return " ".join(str(titulo).lower().rstrip(".").split())

# Trabajos relevantes según los criterios de 14.1 (revisión manual de título y resumen).
RELEVANTES = {
    "predicting unplanned readmission risk in patients with cirrhosis: complication-aware dynamic classifier selection approach":
        ("A", "Shi et al. (2025): META-DES con regiones definidas por complicaciones; readmisión en cirrosis"),
    "the development and validation of a dynamic selection engine to classify chf patient readmission risk using home telemonitoring data":
        ("A", "Motor de selección dinámica para readmisión en insuficiencia cardiaca con telemonitoreo (2015)"),
    "moe-health: a mixture of experts framework for robust multimodal healthcare prediction":
        ("A", "Mezcla de expertos neuronales con compuerta dinámica; incluye readmisión (red neuronal)"),
    "knowledge-augmented patient network embedding-based dynamic model selection for predictive analysis of pediatric drug-induced liver injury":
        ("A", "Huang, Shi et al. (2025): selección dinámica sobre incrustaciones de pacientes; hepatotoxicidad pediátrica y readmisión en insuficiencia cardiaca"),
    "rim-decay: a deep learning method for clinical prediction":
        ("A", "Tesis (2022): red profunda para readmisión a 30 días; la selección dinámica es solo un comparador"),
    "development and validation of a type 2 diabetes machine learning classification model for ehr-based diagnostics and clinical decision support":
        ("B", "Glanz et al. (2022): DES entre otros modelos para diagnóstico de diabetes tipo 2"),
    "development and validation of a type 2 diabetes machine learning classification model for clinical decision support framework":
        ("B", "Glanz et al. (2022), otra versión del mismo preprint"),
    "enhanced diabetic retinopathy diagnosis: a comparative analysis of models with an ensemble classifier and deep q learning":
        ("B", "Selección dinámica de clasificadores para diagnóstico de retinopatía diabética en imágenes"),
    "drecq: design of an efficient model for enhanced diabetic retinopathy diagnosis using ensemble classifiers and deep q learning process":
        ("B", "Mismo trabajo de retinopatía diabética, con otro título"),
    "multi-classifier approaches for supporting clinical decision making":
        ("B", "Groccia et al. (2020): selección dinámica de clasificadores evaluada, entre otros, en un conjunto de retinopatía diabética"),
    "historical segmentation for budgeted preprocessing pipeline selection under distribution shift":
        ("D", "Pote (2026): elige el pipeline por región de la población; evaluado en Diabetes Readmission"),
    "dynamic ensemble selection via optimization of competence regions":
        ("C", "Optimiza la definición de la región de competencia; no usa Brier ni encogimiento"),
    "the limits of patient similarity: graph neural networks versus tabular models for 30-day hospital readmission":
        ("D", "Grafos de similitud entre pacientes en Diabetes 130-US; ninguno supera a Random Forest"),
}

registros["clave"] = registros["título"].map(normalizar)
registros["criterio"] = registros["clave"].map(lambda t: RELEVANTES.get(t, ("", ""))[0])

OBSERVACIONES = {
    "C1": "Sin resultados",
    "C2": "Sin resultados",
    "C3": "Glanz et al. usan DES para diagnóstico de diabetes tipo 2, no readmisión; en el estudio de enfermedad renal la diabetes es solo una variable",
    "C4": "Estudios de readmisión en Diabetes 130-US con ensambles estáticos (apilamiento, votación, XGBoost); ninguno con selección dinámica",
    "C5": "Sin resultados en español",
    "C6": "Solo Shi et al. (cirrosis); el otro registro es de ortopedia",
    "C7": "Medidas de competencia del grupo de Kurzynski (familia RRC) en prótesis; ninguno con encogimiento local-global ni Brier estratificado",
    "C8": "Shi et al. (cirrosis), un motor de selección para insuficiencia cardiaca y una mezcla de expertos neuronal; ninguno en diabetes",
}
# Google Scholar busca en el texto completo: sus observaciones son propias.
OBSERVACIONES_GS = {
    "C1": "Shi et al. (cirrosis), KPNDS (hepatotoxicidad y readmisión en insuficiencia cardiaca), una tesis con DES como comparador y la selección de pipelines por región de Pote (2026); el resto solo menciona los términos en las referencias",
    "C2": "Shi et al. y KPNDS; los otros tres son revisiones o estudios de DES que citan trabajos de readmisión",
    "C3": "Glanz et al. (diabetes tipo 2, dos versiones) y dos de retinopatía diabética, más los ya hallados en C1; el resto usa Pima como banco de prueba o menciona diabetes en las referencias",
    "C4": "Un estudio de detección de atípicos en Diabetes 130-US que elige dinámicamente parámetros de agrupamiento, no clasificadores",
    "C5": "Sin resultados en español",
    "C6": "Solo Shi et al. (cirrosis)",
    "C7": "Trabajos de DES que usan el Brier solo como métrica de evaluación, no como competencia; ninguno con encogimiento",
    "C8": "Shi et al., KPNDS y la tesis con DES como comparador; selección de pipelines por región en Diabetes Readmission (Pote, 2026); ensamble Rashomon estático evaluado en Diabetes Readmission",
}

revisados = registros.groupby(["base", "cadena"]).size().rename("revisados")
relevantes = (registros.query("criterio != ''").groupby(["base", "cadena"])["clave"]
              .nunique().rename("relevantes"))
matriz = (busquedas.set_index(["base", "cadena"])[["resultados"]]
          .join(revisados).join(relevantes).reset_index())
# Una base que no respondió queda sin número de resultados (no como cero).
matriz["resultados"] = matriz["resultados"].astype("Int64")
matriz[["revisados", "relevantes"]] = matriz[["revisados", "relevantes"]].fillna(0).astype(int)
def observacion(base, cadena, n):
    if pd.isna(n):
        return "La base no respondió"
    if base == "Google Scholar":
        return OBSERVACIONES_GS[cadena]
    return OBSERVACIONES[cadena] if n > 0 or cadena in ("C1", "C2", "C5") else "Sin resultados en esta base"

matriz["observación"] = [observacion(b, c, n) for b, c, n in
                         zip(matriz["base"], matriz["cadena"], matriz["resultados"])]
orden = {"OpenAlex": 0, "PubMed": 1, "arXiv": 2, "Semantic Scholar": 3, "Google Scholar": 4}
matriz = matriz.sort_values(["cadena", "base"], key=lambda s: s.map(orden) if s.name == "base" else s)
guardar_resultado(matriz, f"literatura/matriz_novedad_{FECHA}")
with pd.option_context("display.max_colwidth", 120):
    display(matriz.set_index(["cadena", "base"]))

resultados  revisados  relevantes                                                                                                              observación
cadena base                                                                                                                                                                        
C1     OpenAlex                   0          0           0                                                                                                           Sin resultados
       PubMed                     0          0           0                                                                                                           Sin resultados
       arXiv                      0          0           0                                                                                                           Sin resultados
       Semantic Scholar           0          0           0                                                                                                           Sin resultados
       Google Scholar            29         29           4  Shi et al. (cirrosis), KPNDS (hepatotoxicidad y readmisión en insuficiencia cardiaca), una tesis con DES como compar...
C2     OpenAlex                   0          0           0                                                                                                           Sin resultados
       PubMed                     0          0           0                                                                                                           Sin resultados
       arXiv                      0          0           0                                                                                                           Sin resultados
       Semantic Scholar           0          0           0                                                                                                           Sin resultados
       Google Scholar             5          5           2                     Shi et al. y KPNDS; los otros tres son revisiones o estudios de DES que citan trabajos de readmisión
C3     OpenAlex                   3          3           2  Glanz et al. usan DES para diagnóstico de diabetes tipo 2, no readmisión; en el estudio de enfermedad renal la diabe...
       PubMed                     0          0           0                                                                                              Sin resultados en esta base
       arXiv                      0          0           0                                                                                              Sin resultados en esta base
       Semantic Scholar           4          4           1  Glanz et al. usan DES para diagnóstico de diabetes tipo 2, no readmisión; en el estudio de enfermedad renal la diabe...
       Google Scholar           466        464           8  Glanz et al. (diabetes tipo 2, dos versiones) y dos de retinopatía diabética, más los ya hallados en C1; el resto us...
C4     OpenAlex                  23         23           1  Estudios de readmisión en Diabetes 130-US con ensambles estáticos (apilamiento, votación, XGBoost); ninguno con sele...
       PubMed                     4          4           0  Estudios de readmisión en Diabetes 130-US con ensambles estáticos (apilamiento, votación, XGBoost); ninguno con sele...
       arXiv                      0          0           0                                                                                              Sin resultados en esta base
       Semantic Scholar          16         16           1  Estudios de readmisión en Diabetes 130-US con ensambles estáticos (apilamiento, votación, XGBoost); ninguno con sele...
       Google Scholar             1          1           0  Un estudio de detección de atípicos en Diabetes 130-US que elige dinámicamente parámetros de agrupamiento, no clasif...
C5     OpenAlex                   0          0           0       

In [4]:
resumen = matriz.groupby("base")[["resultados", "revisados"]].sum()
resumen["relevantes (únicos)"] = (registros.query("criterio != ''").groupby("base")["clave"].nunique()
                                  .reindex(resumen.index).fillna(0).astype(int))
total_unicos = registros["clave"].nunique()
relevantes_unicos = registros.query("criterio != ''")["clave"].nunique()
display(resumen)
print(f"Registros únicos entre las cinco bases: {total_unicos}; relevantes: {relevantes_unicos}.")

,resultados,revisados,relevantes (únicos)
base,,,
Google Scholar,546,544,8
OpenAlex,39,39,5
PubMed,5,5,1
Semantic Scholar,32,32,4
arXiv,1,1,1


Registros únicos entre las cinco bases: 516; relevantes: 13.


**Cribado de Google Scholar.** En las cadenas con pocos resultados (C1, C2, C4, C6, C7 y C8, con 32 registros como máximo) se revisaron a mano todos los registros. En C3, con 464 registros, la mayoría solo menciona los términos en las referencias, así que se aplicó un filtro automático y se revisó a mano solo lo que pasó. Un registro pasa el filtro si su título o su fragmento contiene, a la vez:
- un término de selección dinámica (*dynamic ensemble/classifier selection*, *dynamic selection*, META-DES, KNORA o región de competencia);
- un término del problema (readmisión o diabetes).

Los candidatos se revisaron a mano por título, fragmento y, cuando hizo falta, resumen. La celda siguiente los lista.

In [5]:
if CARPETA_GS.exists():
    gs = registros.query("base == 'Google Scholar' and cadena == 'C3'").copy()
    texto = (gs["título"] + " " + gs["fragmento"].fillna("")).str.lower()
    seleccion = texto.str.contains(r"dynamic (?:ensemble|classifier|model) selection|dynamic selection|meta-des|knora|regions? of competence|competence region")
    problema = texto.str.contains(r"readmi|rehospitali|diabet|130-us")
    candidatos = gs[seleccion & problema].drop_duplicates("clave")
    print(f"Registros de Google Scholar en C3: {gs['clave'].nunique()}; pasan el filtro: {len(candidatos)}; "
          f"relevantes tras la revisión manual: {candidatos['criterio'].ne('').sum()}")
    with pd.option_context("display.max_colwidth", 110):
        display(candidatos.assign(relevante=candidatos["criterio"].replace("", "no"))
                [["año", "título", "relevante"]].sort_values(["relevante", "año"]).reset_index(drop=True))

Registros de Google Scholar en C3: 458; pasan el filtro: 54; relevantes tras la revisión manual: 5


,año,título,relevante
0,2025.0,Predicting Unplanned Readmission Risk in Patients With Cirrhosis: Complication-Aware Dynamic Classifier Se...,A
1,2020.0,Multi-classifier approaches for supporting clinical decision making,B
2,2022.0,Development and validation of a type 2 diabetes machine learning classification model for EHR-based diagno...,B
3,2022.0,Development and validation of a type 2 diabetes machine learning classification model for clinical decisio...,B
4,2024.0,Enhanced Diabetic Retinopathy Diagnosis: A Comparative Analysis of models with an Ensemble classifier and ...,B
5,2003.0,Diversity in Ensemble Feature Selection [R],no
6,2008.0,Empirical analysis of reliability estimates for individual regression predictions,no
7,2009.0,Ensemble learning: A study on different variants of the dynamic selection approach,no
8,2011.0,Classifier ensemble construction with rotation forest to improve medical diagnosis performance of machine ...,no
9,2012.0,Retinal vessels segmentation using supervised classifiers decisions fusion,no


**Lectura.** En las cuatro bases que buscan en título y resumen, las cadenas que combinan directamente el modelo con el problema (C1, C2 y C5) no devuelven ningún resultado. Google Scholar, que busca en el texto completo, sí devuelve resultados para C1 y C2, y la revisión los reduce a tres trabajos:
- **Shi et al. (2025).** Readmisión en cirrosis.
- **KPNDS (Huang, Shi et al., 2025).** Del mismo grupo, aplicado a hepatotoxicidad pediátrica, con una validación adicional en readmisión por insuficiencia cardiaca.
- **Una tesis de 2022.** Usa la selección dinámica solo como comparador de una red profunda.

Ampliar el modelo a cualquier método de selección dinámica (C3) hace aparecer aplicaciones en diabetes, pero todas son de **diagnóstico**: diabetes tipo 2 o retinopatía diabética. La cadena sobre el conjunto de datos (C4) y los estudios de readmisión en Diabetes 130-US usan modelos o ensambles **estáticos**, con la misma regla de combinación para todos los pacientes. La única excepción local es la selección de pipelines por región de Pote (2026), que no estima la competencia de cada modelo por paciente.

## 14.4. Búsqueda por citación (*snowballing*)

Se partió de cinco trabajos semilla, los más cercanos a DES-CPL por problema o por método, más uno sobre los mismos datos:

- **hacia adelante:** todos los trabajos que citan a la semilla, filtrados por readmisión o diabetes en título o resumen;
- **hacia atrás:** todas las referencias de la semilla, revisadas por título.

La consulta se hizo en OpenAlex el mismo día.

| Semilla | Por qué | Citas totales | Adelante con readmisión o diabetes | Referencias revisadas | Nuevos relevantes |
|---|---|---|---|---|---|
| Shi et al. (2025), selección dinámica en readmisión por cirrosis | El más cercano por problema | 1 | 1 (revisión de IA en hepatología) | 41 | 0 |
| Cruz, Sabourin y Cavalcanti (2018), revisión de selección dinámica | Revisión de referencia del área | 465 | 4 (Shi et al.; tres sin readmisión) | 188 | 0 |
| Cruz et al. (2015), META-DES | Método usado por Shi et al. | 261 | 4 (Glanz et al., en dos versiones; dos sin readmisión) | 42 | 0 |
| Cruz et al. (2020), DESlib | Biblioteca de referencia; quien aplica DES suele citarla | 86 | 2 (hipoglucemia en diabetes tipo 1; enfermedad renal) | 17 | 1 parcial |
| Selección dinámica por optimización de regiones de competencia (2026) | El más cercano por método reciente | 1 | 0 | 54 | 0 |
| Límites de la similitud entre pacientes (2026) | Enfoque local en los mismos datos | 0 | 0 | 20 | 0 |

**Hallazgos.**
- **Por citación hacia adelante** apareció un solo trabajo nuevo: Lara-Abelenda et al. (2025), que cita DESlib en un estudio de fusión multimodal para predecir **hipoglucemia grave** en diabetes tipo 1. Es criterio B parcial: población con diabetes, pero otro desenlace, datos multimodales y la selección dinámica no es su propuesta.
- **Por citación hacia atrás** no apareció ningún trabajo de selección dinámica en readmisión distinto de los ya encontrados. En las referencias de Shi et al. figuran:
  - un ensamble conjunto de *boosting* y apilamiento para readmisión (Yu et al., 2019), que es estático;
  - un modelo de decisión con selección dinámica de clasificadores (Xu y Fu, 2020), que es de toma de decisiones multicriterio y no clínico.

## 14.5. Repositorios de código y bancos de prueba

**GitHub** (búsqueda de repositorios en la API pública, el mismo día):

| Consulta | Repositorios | Observación |
|---|---|---|
| `"dynamic ensemble selection"` | 29 | Bibliotecas (DESlib, deskit), variantes del método y aplicaciones a spam, noticias falsas, energía solar y flujos de datos. El único clínico valida un DES mejorado para recurrencia de linfoma con datos de glioma, insuficiencia cardiaca y hepatitis |
| `readmission dynamic selection` | 0 | — |
| `diabetes readmission` | 1 422 | Proyectos con regresión logística, Random Forest, XGBoost y redes; ninguno de los 100 más relevantes menciona selección dinámica |

**Bancos de prueba.** Los estudios de selección dinámica se evalúan sobre colecciones de UCI y KEEL (Cruz et al., 2018). La única base con «diabetes» en esas colecciones es Pima, que es de diagnóstico. Diabetes 130-US no aparece en ellas.

## 14.6. Búsqueda por problema en Web of Science y Scopus

El 30 de octubre de 2026, antes de elegir el modelo original, se exportaron de Web of Science y Scopus los artículos sobre readmisión hospitalaria de pacientes con diabetes: 257 y 83 registros, respectivamente. También se exportó una búsqueda general de modelos de clasificación. Con ese cruce se armó la lista de modelos que no aparecen en la literatura del problema, de la que salió la familia de selección dinámica (capítulo 13, sección 13.2.1).

Ninguno de esos 340 registros usa selección dinámica de clasificadores o de ensambles. Esta búsqueda complementa a la de 14.2 desde el lado del problema: aquella parte del modelo y esta del problema, y ninguna de las dos encuentra la combinación.

## 14.7. Trabajos más cercanos y en qué se diferencia DES-CPL

| Trabajo | Problema | Método | Diferencia con DES-CPL |
|---|---|---|---|
| Shi et al. (2025), *JMIR Med Inform* | Readmisión no planeada en **cirrosis** (historias clínicas de China) | META-DES con regiones de competencia definidas por las complicaciones del paciente | Otra enfermedad y otros datos. La competencia es la exactitud estimada por un metaclasificador, no la calidad probabilística (Brier). No estratifica la región por clase, no excluye encuentros del mismo paciente y no encoge hacia la competencia global |
| KPNDS: Huang, Shi et al. (2025), *IEEE Trans. Biomed. Eng.* | **Hepatotoxicidad** pediátrica por fármacos y, como validación adicional, readmisión a 28 días en **insuficiencia cardiaca** | Selección dinámica por metaaprendizaje sobre incrustaciones de pacientes en una red aumentada con conocimiento biomédico; los metaclasificadores son redes (MLP, Transformer, KAN) | Otras enfermedades. La competencia la predice un metaclasificador neuronal sobre un grafo de conocimiento; en DES-CPL se estima con el Brier en un vecindario de perfiles de salida. No usa encogimiento local-global |
| Tesis de 2022 (Basnet), *Rim-Decay* | Readmisión a 30 días (población no especificada en el fragmento) | Red profunda propuesta; la selección dinámica aparece solo como comparador | La selección dinámica no es la propuesta ni se adapta al problema |
| Motor de selección dinámica para insuficiencia cardiaca (2015) | Readmisión en **insuficiencia cardiaca** con telemonitoreo domiciliario | Selección dinámica de clasificadores (solo se accedió al título) | Otra enfermedad y datos de sensores en el hogar, no administrativos. No se encontró el texto para comparar el método |
| Glanz et al. (2022), preprint | **Diagnóstico** de diabetes tipo 2 con pruebas de laboratorio | DES como uno de varios modelos comparados | Otro desenlace: diagnóstico, no readmisión. DES se usa como herramienta estándar y no se modifica |
| Retinopatía diabética: DRECQ (2024) y Groccia et al. (2020) | **Diagnóstico** de retinopatía diabética (imágenes o un conjunto de prueba clínico) | Selección dinámica de clasificadores, combinada con aprendizaje por refuerzo en DRECQ | Otro desenlace y otro tipo de datos; no es readmisión |
| Lara-Abelenda et al. (2025), *Eng. Appl. AI* | **Hipoglucemia grave** en diabetes tipo 1 | Fusión temprana y tardía multimodal; cita DESlib | Otro desenlace y datos multimodales. La selección dinámica no es la contribución |
| Límites de la similitud entre pacientes (2026), IDAP | Readmisión a 30 días en **Diabetes 130-US** | Grafos de similitud entre pacientes, GraphSAGE y suavizado sobre el grafo | Usa la vecindad entre pacientes para predecir, no para elegir modelos. Su Random Forest ajustado (AUC-PR 0.215) supera a todos los métodos de grafos, un resultado coherente con el techo que encontramos en el capítulo 13 |
| Pote (2026), tesis de Purdue | Selección de pipelines bajo cambio de distribución, evaluada en **Diabetes Readmission** y Bank Marketing | Segmenta los datos históricos en regiones y elige el pipeline de preprocesamiento y el clasificador de cada región con un presupuesto de tres evaluaciones | Es la idea más cercana en los mismos datos, pero la región es un segmento de la población y no el vecindario de cada paciente. Elige un pipeline por validación, no pondera modelos por una competencia probabilística local |
| Zuin y Veloso (2025), *ACM TKDD* | Varios conjuntos, entre ellos **Diabetes Readmission** | Ensamble Rashomon: agrupa modelos de desempeño similar por sus explicaciones y elige un subconjunto diverso | La selección es **estática**: el mismo ensamble para todos los pacientes |
| MoE-Health (2025), arXiv | Predicción clínica multimodal, incluida la readmisión | Mezcla de expertos neuronales con compuerta dinámica | Red neuronal, excluida por la guía como modelo original. La compuerta se aprende por gradiente, no se estima a partir de la competencia local |
| Selección por optimización de regiones de competencia (2026), *Appl. Soft Comput.* | Bancos de prueba generales | Optimiza la región de competencia | No aplicado a readmisión ni a diabetes. Sin resumen accesible para comparar la medida de competencia |
| BPE, *Behavioral Profiling Ensemble* (2026), arXiv | Bancos de prueba generales | Pondera cada modelo según su perfil de comportamiento propio, sin conjunto de referencia | En BPE el perfil es de cada modelo. En DES-CPL el perfil es de cada paciente: el vector de logits que define su vecindario |

**Antecedentes metodológicos de cada componente** (detallados en 13.2.1):
- KNOP, para la región en el espacio de salidas;
- RRC (Woloszynski y Kurzynski, 2011), para la competencia probabilística;
- la selección dinámica de regresores por error, para la competencia como pérdida;
- KNORA-B y DES-MI (García et al., 2018), para las regiones con ambas clases.

Ninguno de estos trabajos ni de los anteriores usa el **encogimiento de la competencia local hacia la global** con un λ elegido en los datos, ni la **reponderación del Brier por la prevalencia local estimada**: en la búsqueda C7 no aparecieron.

## 14.8. Declaración de novedad

> Al mejor de nuestro conocimiento y según la búsqueda descrita, que abarca cinco bases (OpenAlex, PubMed, arXiv, Semantic Scholar y Google Scholar) consultadas el 10 de noviembre de 2026 con ocho cadenas en inglés y español, búsqueda por citación desde seis trabajos semilla, repositorios de GitHub y la búsqueda por problema en Web of Science y Scopus, **no se encontraron estudios que apliquen selección dinámica de clasificadores o de ensambles a la predicción de la readmisión hospitalaria de pacientes con diabetes**.
>
> Los trabajos más cercanos aplican selección dinámica a la readmisión en **otras enfermedades**, a otros desenlaces en diabetes o vecindarios de pacientes a los mismos datos. Son, respectivamente:
> - en otras enfermedades: Shi et al. (2025) en cirrosis; el marco KPNDS del mismo grupo (2025), en hepatotoxicidad pediátrica y readmisión por insuficiencia cardiaca; y un motor de selección de 2015 para insuficiencia cardiaca;
> - en diabetes, con otros desenlaces: Glanz et al. (2022), diagnóstico de diabetes tipo 2; dos trabajos de retinopatía diabética; y Lara-Abelenda et al. (2025), hipoglucemia en diabetes tipo 1;
> - en los mismos datos: el estudio de similitud entre pacientes de 2026, que no selecciona modelos, y la selección de pipelines por región de Pote (2026), que elige un pipeline por segmento de la población y no por paciente.
>
> Además, no se encontró en la literatura de selección dinámica la combinación de cuatro elementos que define a DES-CPL:
> - competencia medida con el puntaje de Brier, reponderado por la prevalencia local;
> - región estratificada por clase en el espacio de logits de los modelos base, que excluye los encuentros del mismo paciente;
> - encogimiento de la competencia local hacia la global;
> - poda previa.
>
> Cada componente por separado tiene antecedentes, que se citan en la sección 14.7.

**Alcance de la afirmación.**
- La novedad es de **aplicación y de combinación**, no de familia: la selección dinámica de ensambles existe desde hace más de veinte años.
- El capítulo 13 muestra que DES-CPL no supera al mejor benchmark. La novedad no depende del resultado, pero el resultado limita la nota al tope de 4.0 (sección 13.8).

## 14.9. Limitaciones de la revisión y repetición

- **Google Scholar** se consultó a mano y se cribó con un filtro automático sobre título y fragmento. Un trabajo cuyo fragmento visible no mencione a la vez la selección dinámica y el problema pudo pasar el filtro sin ser detectado. En las cadenas C1, C2, C4, C6, C7 y C8 se revisaron todos los registros; en C3 se copiaron 464 de los 466 resultados.
- **Web of Science y Scopus** se usaron en la búsqueda por problema, pero no con las ocho cadenas del modelo. OpenAlex, Semantic Scholar y Google Scholar indexan buena parte de esas colecciones.
- **Revisión por título y resumen.** Algunos trabajos se compararon solo por título, fuente y fragmento: los que no tienen resumen accesible (el motor de 2015 y el de regiones de competencia de 2026) y la tesis de 2022.
- **Literatura gris.** Las búsquedas cubren artículos, preprints y repositorios, pero no tesis ni informes técnicos no indexados.
- **Repetición.** La guía pide repetir la búsqueda antes de la entrega final:
  - Para las cuatro bases con API basta con correr `python experiments/revision_literatura.py`. Guarda los archivos con la fecha del día sin borrar los anteriores, y la celda siguiente compara las dos consultas cuando existen.
  - Para Google Scholar se vuelven a copiar las páginas de resultados.

In [6]:
if len(fechas) > 1:
    anterior = pd.read_csv(LITERATURA / f"busquedas_{fechas[-2]}.csv")
    comparacion = (anterior.set_index(["base", "cadena"])["resultados"].rename(fechas[-2]).to_frame()
                   .join(busquedas.set_index(["base", "cadena"])["resultados"].rename(fechas[-1])))
    comparacion["cambio"] = comparacion[fechas[-1]] - comparacion[fechas[-2]]
    display(comparacion[comparacion["cambio"] != 0] if (comparacion["cambio"] != 0).any() else comparacion)
    nuevos = set(registros["clave"]) - set(pd.read_csv(LITERATURA / f"registros_{fechas[-2]}.csv")["título"].map(normalizar))
    print(f"Registros nuevos desde {fechas[-2]}: {len(nuevos)}")
    for t in sorted(nuevos):
        print(" -", t)
else:
    print(f"Solo hay una consulta ({FECHA}). Repetir antes de la entrega final.")

Solo hay una consulta (2026-11-10). Repetir antes de la entrega final.
